# Rust 104: Strings — Practice

Companion exercises for `04_strings_e.ipynb`. Every section mirrors the section of the same number in the example notebook, and there is one exercise for each of its demonstration cells, so a `**See**` reference lands on the exact code that shows the technique.

The checks at the bottom of each cell are the grader. Edit only the parts marked `TODO`; when a cell runs without panicking you will see an `Exercise N.M passed` line.


### Contents

- [0. How to Complete These Exercises](#0-how-to-complete-these-exercises)
- [1. `String` and `&str`: Two Types, One Buffer](#1-string-and-str-two-types-one-buffer)
- [2. UTF-8: Bytes, Chars and Boundaries](#2-utf-8-bytes-chars-and-boundaries)
- [3. Slicing, and Why It Can Panic](#3-slicing-and-why-it-can-panic)
- [4. Literals: Escapes, Raw Strings, Multi-Line](#4-literals-escapes-raw-strings-multi-line)
- [5. Growing and Shrinking a `String`](#5-growing-and-shrinking-a-string)
- [6. Iterating: `bytes`, `chars`, `char_indices`](#6-iterating-bytes-chars-char_indices)
- [7. The Method Inventory](#7-the-method-inventory)
- [8. Splitting, Lines and Joining](#8-splitting-lines-and-joining)
- [9. Converting Between Bytes and Text](#9-converting-between-bytes-and-text)
- [10. Passing Strings: `&str` in, `String` out](#10-passing-strings-str-in-string-out)
- [11. Text Processing Patterns](#11-text-processing-patterns)
- [12. Formatting and Display](#12-formatting-and-display)
- [13. Reading the Compiler on Strings](#13-reading-the-compiler-on-strings)
- [14. Unicode Beyond ASCII](#14-unicode-beyond-ascii)
- [15. Performance Notes for Low-Latency Code](#15-performance-notes-for-low-latency-code)
- [16. Review: Mixed Problems](#16-review-mixed-problems)

§1–§15 each have one exercise per demonstration cell of the example notebook; §16 mixes them. When a check fails, the fastest route back is §16 of the example notebook, which is the method inventory in one page.


## 0. How to Complete These Exercises

This notebook runs on the same **evcxr** kernel as the example notebook. Three consequences shape how you work:

1. **No `fn main`.** The body of a cell *is* the body of `main`. Type statements directly.
2. **State persists.** A binding from one cell is visible in every cell below it.
3. **Items are order-independent.** A `fn`, a `const` or a `use` line may be used above the cell that defines it.

A fourth consequence bites harder in this chapter than anywhere else in the track:

> **Item names must be unique across the whole notebook.** Every cell shares one namespace, so two functions with the same name are a redefinition error, even in different sections. If the compiler says a name "is defined multiple times", it is pointing at an *earlier* cell. Rename the new one.

### The stub convention

Each exercise cell declares a signature, a doc comment stating the contract, and a `// --- check (do not modify) ---` block of `assert_eq!`/`assert!` calls. The body you must fill in is `todo!()`, which compiles but **panics** when executed:

```text
thread '<main>' panicked at ... not yet implemented
```

Replace the `todo!()` and re-run the cell. An `Exercise N.M passed` line means you are done.

### Two notes specific to strings

- The checks compare against literal expected values, and many of them contain non-ASCII text. If a check fails, print `len()` and `chars().count()` for both sides before assuming the check is wrong.
- A failing check aborts the rest of *that cell* only. The kernel survives, and the next cell still runs.

### Expect warnings until you finish

Every unsolved stub whose parameters are untouched emits `unused variable`, so a fresh notebook shows a long list of them. That is normal and self-cleaning: each warning disappears when you implement the body. If a diagnostic appears that is not in that list, it is yours.

**See**: §0 and §13 of the example notebook.


### Exercise 0.1 — The kernel is alive

**Goal**: prove a cell compiles, runs and reports a result.

**Given**: nothing; this is the smoke test.

**Objectives**

- implement `byte_count` so it returns the number of **bytes** in `text`
- keep the signature unchanged

**See**: §0.1 and §2.1 of the example notebook.

**Hint**: a `&str` carries its byte length with it; no loop is needed.


In [ ]:
/// Return the number of **bytes** in `text` (not characters).
///
/// TODO: replace `todo!()` with the byte length.
fn byte_count(text: &str) -> usize {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(byte_count(""), 0);
assert_eq!(byte_count("hello"), 5);
assert_eq!(byte_count("héllo"), 6);
println!("Exercise 0.1 passed");


## 1. `String` and `&str`: Two Types, One Buffer

### Exercise 1.1 — Own the text

**Goal**: produce an owned `String` from a borrowed one.

**Info**: A `-> String` function must hand back ownership of a fresh value. `String::from(text)` and `text.to_owned()` both do that; the `&str` parameter does not.

**Objectives**

- return an owned copy of `text`
- keep the signature unchanged

**See**: §1.2 of the example notebook.

**Hint**: `String::from` is the shortest spelling.


In [ ]:
/// Return an owned copy of `text`.
///
/// TODO: replace `todo!()` with an owned copy.
fn owned_copy(text: &str) -> String {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(owned_copy("hello"), "hello");
assert_eq!(owned_copy(""), "");
assert_eq!(owned_copy("héllo").len(), 6);
println!("Exercise 1.1 passed");


### Exercise 1.2 — Compare across the boundary

**Goal**: compare a `&str` with a `String` without converting either one.

**Info**: `PartialEq` is implemented in both directions between `str` and `String`, so the comparison crosses the type boundary on its own.

**Objectives**

- return `true` when `borrowed` and `owned` hold the same bytes
- keep the signature unchanged

**See**: §1.3 of the example notebook.

**Hint**: one `==` is enough. The two sides do not have to be the same type.


In [ ]:
/// Compare a `&str` with a `&String`.
///
/// TODO: replace `todo!()` with the comparison.
fn same_text(borrowed: &str, owned: &String) -> bool {
    todo!()
}

// --- check (do not modify) ---
assert!(same_text("same", &String::from("same")));
assert!(!same_text("same", &String::from("other")));
assert!(same_text("", &String::new()));
println!("Exercise 1.2 passed");


### Exercise 1.3 — Bytes and characters coincide, until they do not

**Goal**: test the assumption that hides most string bugs.

**Info**: For ASCII text every character is one byte, so `len()` and the character count agree. As soon as one non-ASCII character appears — here `é`, which is two bytes — they stop agreeing.

**Objectives**

- return `true` exactly when the byte length equals the character count
- keep the signature unchanged

**See**: §1.4 and §2.1 of the example notebook.

**Hint**: two length calls, compared. One of them walks the string.


In [ ]:
/// Return true when `text`'s byte length equals its character count.
///
/// TODO: replace `todo!()` with the comparison.
fn ascii_lengths_agree(text: &str) -> bool {
    todo!()
}

// --- check (do not modify) ---
assert!(ascii_lengths_agree("hello"));
assert!(ascii_lengths_agree(""));
assert!(!ascii_lengths_agree("héllo"));
assert!(ascii_lengths_agree("12345"));
println!("Exercise 1.3 passed");


### Exercise 1.4 — The legal spelling of "index 0"

**Goal**: get the first piece of a string out of it.

**Info**: `s[0]` does not compile: `str` is indexed by **ranges** of `usize`, not by a single index. A range whose ends are char boundaries is the legal spelling, and elision gives the returned slice the lifetime of the argument.

**Objectives**

- return the slice covering the first byte of `text`
- keep the signature unchanged

**See**: §1.5 of the example notebook.

**Hint**: a half-open range, borrowed out of `text`.


In [ ]:
/// Return the first byte of `text`.
///
/// TODO: replace `todo!()` with a byte-range slice.
fn first_byte_slice(text: &str) -> &str {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(first_byte_slice("hello"), "h");
assert_eq!(first_byte_slice("héllo"), "h");
assert_eq!(first_byte_slice("7"), "7");
println!("Exercise 1.4 passed");


### Exercise 1.5 — A literal is borrowed and `'static`

**Goal**: return a view of a literal rather than an owned copy.

**Info**: A string literal is `&'static str`: a pointer into the program's read-only data. No allocation happens, and the value can be returned from a function that owns nothing.

**Objectives**

- return the literal `immutable text` as a `&'static str`
- do not allocate, and do not use `String`
- keep the signature unchanged

**See**: §1.1 of the example notebook.

**Hint**: the body is the literal itself.


In [ ]:
/// Return the literal `"immutable text"` as a `&'static str`.
///
/// TODO: replace `todo!()` with the literal.
fn literal_view() -> &'static str {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(literal_view(), "immutable text");
assert_eq!(literal_view().len(), 14);
assert_eq!(literal_view().chars().count(), 14);
println!("Exercise 1.5 passed");


## 2. UTF-8: Bytes, Chars and Boundaries

### Exercise 2.1 — Count characters, not bytes

**Goal**: count what a user would call characters.

**Info**: `len()` is stored in the `&str` and is O(1). The character count requires walking the encoding, because a character is 1 to 4 bytes wide.

**Objectives**

- return the number of scalar values in `text`
- keep the signature unchanged

**See**: §2.1 of the example notebook.

**Hint**: `chars()` yields them; there is a counting method on the iterator.


In [ ]:
/// Count the characters in `text` (not the bytes).
///
/// TODO: replace `todo!()` with the character count.
fn char_count(text: &str) -> usize {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(char_count("hello"), 5);
assert_eq!(char_count("héllo"), 5);
assert_eq!(char_count("👋"), 1);
assert_eq!(char_count(""), 0);
println!("Exercise 2.1 passed");


### Exercise 2.2 — How wide is a character?

**Goal**: report the encoded width of one character.

**Info**: `char` is a Unicode scalar value, so its width is 1 to 4 bytes. The method that reports it is named for the encoding, because a bare `len` would be ambiguous.

**Objectives**

- return the number of bytes `c` occupies in UTF-8
- keep the signature unchanged

**See**: §2.2 of the example notebook.

**Hint**: `char` has a method whose name ends in `_utf8`.


In [ ]:
/// Return how many bytes `c` occupies in UTF-8.
///
/// TODO: replace `todo!()` with the encoded width.
fn utf8_width(c: char) -> usize {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(utf8_width('a'), 1);
assert_eq!(utf8_width('é'), 2);
assert_eq!(utf8_width('€'), 3);
assert_eq!(utf8_width('👋'), 4);
println!("Exercise 2.2 passed");


### Exercise 2.3 — Where the characters are

**Goal**: pair each character with the byte offset it starts at.

**Info**: `char_indices()` is the only correct way to build a byte offset for slicing. The offsets it yields are char boundaries by construction.

**Objectives**

- return every `(byte offset, char)` pair of `text`, in order
- keep the signature unchanged

**See**: §2.3 of the example notebook.

**Hint**: one iterator, collected into the pair type.


In [ ]:
/// Return every `(byte offset, char)` pair in `text`.
///
/// TODO: replace `todo!()` with the collected pairs.
fn char_offsets(text: &str) -> Vec<(usize, char)> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(char_offsets("aé"), vec![(0, 'a'), (1, 'é')]);
assert_eq!(char_offsets("héllo").len(), 5);
assert_eq!(char_offsets(""), Vec::new());
println!("Exercise 2.3 passed");


### Exercise 2.4 — nth character, without an index operator

**Goal**: look up a character by position.

**Info**: Because the encoding is variable-width, the *n*-th character cannot be found without walking the preceding ones. Rust makes that cost visible by not giving you `s[n]`.

**Objectives**

- return the character at character index `n`, or `None` when `text` is shorter
- keep the signature unchanged

**See**: §2.4 of the example notebook.

**Hint**: an iterator adaptor that takes a position and returns an `Option`.


In [ ]:
/// Return the `n`th character of `text`, or `None` if it is shorter.
///
/// TODO: replace `todo!()` with the lookup.
fn nth_char(text: &str, n: usize) -> Option<char> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(nth_char("héllo", 0), Some('h'));
assert_eq!(nth_char("héllo", 1), Some('é'));
assert_eq!(nth_char("héllo", 99), None);
assert_eq!(nth_char("", 0), None);
println!("Exercise 2.4 passed");


## 3. Slicing, and Why It Can Panic

### Exercise 3.1 — Slice by byte offsets

**Goal**: take a byte range out of a string.

**Info**: `&text[start..end]` slices by byte offsets and panics when either end is not a char boundary. This exercise only uses offsets that are boundaries, so the panic-free path is the one being practised.

**Objectives**

- return `text` sliced from byte `start` up to byte `end`
- keep the signature unchanged

**See**: §3.1 of the example notebook.

**Hint**: one range expression, borrowed.


In [ ]:
/// Return the byte range `start..end` of `text`.
///
/// TODO: replace `todo!()` with the slice.
fn byte_slice(text: &str, start: usize, end: usize) -> &str {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(byte_slice("héllo", 0, 1), "h");
assert_eq!(byte_slice("héllo", 3, 6), "llo");
assert_eq!(byte_slice("abc", 1, 1), "");
assert_eq!(byte_slice("h€llo", 1, 4), "€");
println!("Exercise 3.1 passed");


### Exercise 3.2 — Detect the panic without dying

**Goal**: observe a non-boundary slice as a value instead of a crash.

**Info**: `std::panic::catch_unwind` runs a closure and returns whether it panicked. The default panic hook prints to stderr, so a quiet version is installed for the duration and restored afterwards.

**Objectives**

- return `true` when slicing `text` at byte offset 2 panics
- restore the previous panic hook before returning
- keep the signature unchanged

**See**: §3.2 of the example notebook.

**Hint**: `catch_unwind`, then ask the returned result whether it is an error.


In [ ]:
/// Return true when slicing `text` at byte offset 2 panics — that is, when byte 2 is
/// not a char boundary.
///
/// TODO: replace `todo!()` with the caught panic check.
fn slicing_at_two_panics(text: &str) -> bool {
    todo!()
}

// --- check (do not modify) ---
assert!(slicing_at_two_panics("héllo"));
assert!(!slicing_at_two_panics("hello"));
assert!(!slicing_at_two_panics("éé"));
println!("Exercise 3.2 passed");


### Exercise 3.3 — The input-facing form

**Goal**: slice without the possibility of a panic.

**Info**: `str::get(range)` returns `Option<&str>`. This is the form to use when the offsets come from outside the program, where a malformed range should be handled rather than crash the process.

**Objectives**

- return `text[start..end]`, or `""` when that range is out of bounds or not on char boundaries
- keep the signature unchanged

**See**: §3.3 of the example notebook.

**Hint**: `get` plus a default for the `None` arm. The empty string literal is the natural default.


In [ ]:
/// Return `text[start..end]`, or `""` when the range is unusable.
///
/// TODO: replace `todo!()` with the non-panicking lookup.
fn slice_or_empty(text: &str, start: usize, end: usize) -> &str {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(slice_or_empty("héllo", 0, 1), "h");
assert_eq!(slice_or_empty("héllo", 0, 2), "");
assert_eq!(slice_or_empty("abc", 0, 9), "");
assert_eq!(slice_or_empty("abc", 3, 3), "");
println!("Exercise 3.3 passed");


### Exercise 3.4 — A live slice blocks mutation

**Goal**: hit the borrow rule that slicing creates, and satisfy it.

**Info**: A slice keeps its `String` immutably borrowed for as long as the slice lives, so mutating the `String` while the slice is alive is `E0502`. Copying the slice into an owned `String` ends the borrow.

**Objectives**

- copy the first two bytes of `text` into an owned `String`
- then append `!` to `text`
- return the copy, so that `text` ends up as the original plus `!`
- keep the signature unchanged

**See**: §3.4 of the example notebook.

**Hint**: `to_string` (or `to_owned`) on the slice ends the borrow before the push.


In [ ]:
/// Copy the first two bytes of `text` out, append `'!'` to `text`, and return the copy.
///
/// TODO: replace `todo!()` — order matters, because the slice borrows `text`.
fn head_then_append(text: &mut String) -> String {
    todo!()
}

// --- check (do not modify) ---
let mut first = String::from("hello");
assert_eq!(head_then_append(&mut first), "he");
assert_eq!(first, "hello!");
let mut second = String::from("ab");
assert_eq!(head_then_append(&mut second), "ab");
assert_eq!(second, "ab!");
println!("Exercise 3.4 passed");


## 4. Literals: Escapes, Raw Strings, Multi-Line

### Exercise 4.1 — Write the escapes

**Goal**: put a tab, a newline and a backslash into a string literal.

**Info**: `\t`, `\n`, `\\`, `\"`, `\0` and `\u{...}` are the escapes. There is no `\a`, no octal escape and no `\e`, so C source does not port over unedited.

**Objectives**

- return a two-line report whose first line is `id`, a tab, then `name`
- whose second line is `path C:\DEV` — one real backslash
- keep the signature unchanged

**See**: §4.1 of the example notebook.

**Hint**: three escapes are needed: tab, newline and a literal backslash.


In [ ]:
/// Return `"id\tname\npath C:\\DEV"`, written with escapes.
///
/// TODO: replace `todo!()` with the escaped literal.
fn escaped_report() -> &'static str {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(escaped_report(), "id\tname\npath C:\\DEV");
assert_eq!(escaped_report().lines().count(), 2);
assert_eq!(escaped_report().matches('\t').count(), 1);
assert_eq!(escaped_report().matches('\\').count(), 1);
println!("Exercise 4.1 passed");


### Exercise 4.2 — Raw strings

**Goal**: write text that contains backslashes and quotes without escaping.

**Info**: `r"..."` disables escape processing. When the content contains a quote, delimit with hashes: `r#"..."#`. A raw string cannot contain the sequence that terminates it.

**Objectives**

- implement `dev_path` so it returns `C:\DEV\Rust` as a raw string literal
- implement `quoted` so it returns `say "hi"` as a raw string literal with hashes
- keep both signatures unchanged

**See**: §4.2 of the example notebook.

**Hint**: `r"..."` for the path; `r#"..."#` for the one containing quotes.


In [ ]:
/// Return the path `C:\DEV\Rust` as a raw string literal.
///
/// TODO: replace `todo!()` with the raw literal.
fn dev_path() -> &'static str {
    todo!()
}

/// Return `say "hi"` as a raw string literal delimited with hashes.
///
/// TODO: replace `todo!()` with the hashed raw literal.
fn quoted() -> &'static str {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(dev_path(), r"C:\DEV\Rust");
assert_eq!(dev_path().matches('\\').count(), 2);
assert_eq!(quoted(), "say \"hi\"");
assert_eq!(quoted().matches('"').count(), 2);
println!("Exercise 4.2 passed");


### Exercise 4.3 — Multi-line literals and continuation

**Goal**: control whether the source layout shows up in the value.

**Info**: A newline inside a literal is part of the string. Ending a source line with a backslash removes that newline *and* the leading whitespace of the next line, which is the one place where a literal's value depends on layout.

**Objectives**

- return `first second` — one space between the two words
- write it as a literal that spans two source lines, using a continuation backslash
- keep the signature unchanged

**See**: §4.3 of the example notebook.

**Hint**: the break goes immediately after `first `, and the next line is indented.


In [ ]:
/// Return `"first second"` from a literal that spans two source lines.
///
/// TODO: replace `todo!()` with the continued literal.
fn continued() -> &'static str {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(continued(), "first second");
assert_eq!(continued().len(), 12);
assert_eq!(continued().lines().count(), 1);
assert_eq!(continued().split_whitespace().count(), 2);
println!("Exercise 4.3 passed");


### Exercise 4.4 — Byte-string literals

**Goal**: produce bytes rather than text.

**Info**: `b"abc"` has type `&[u8; 3]` — a byte array with no UTF-8 guarantee. It is the right type for protocol headers and binary magic numbers.

**Objectives**

- return the three bytes `abc` as a byte slice
- keep the signature unchanged

**See**: §4.4 of the example notebook.

**Hint**: the `b` prefix on the literal; the return type coerces the array reference to a slice.


In [ ]:
/// Return the bytes `abc` as a `&[u8]`.
///
/// TODO: replace `todo!()` with the byte literal.
fn magic_bytes() -> &'static [u8] {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(magic_bytes(), b"abc");
assert_eq!(magic_bytes().len(), 3);
assert_eq!(magic_bytes(), "abc".as_bytes());
assert_eq!(magic_bytes()[0], 97);
println!("Exercise 4.4 passed");


## 5. Growing and Shrinking a `String`

### Exercise 5.1 — Build a string from pieces

**Goal**: use the two append methods, and feel the difference between them.

**Info**: `push` takes a `char` and `push_str` takes a `&str`. That split is deliberate, and mixing them up produces a distinctive diagnostic.

**Objectives**

- start from an empty `String`
- append `Hello`, then `, world` with `push_str`, then `!` with `push`
- return the result
- keep the signature unchanged

**See**: §5.1 of the example notebook.

**Hint**: one `push_str` call per piece of text, one `push` call for the character.


In [ ]:
/// Build `"Hello, world!"` from three pieces.
///
/// TODO: replace `todo!()` with the built string.
fn greeting() -> String {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(greeting(), "Hello, world!");
assert_eq!(greeting().len(), 13);
assert_eq!(greeting().chars().count(), 13);
println!("Exercise 5.1 passed");


### Exercise 5.2 — Allocate once

**Goal**: create a buffer with room already reserved.

**Info**: A new `String` starts with capacity 0, so the first push allocates and later pushes may reallocate. Reserving up front turns a growing loop into a single allocation.

**Objectives**

- return an empty `String` whose capacity is at least `bytes`
- keep the signature unchanged

**See**: §5.2 of the example notebook.

**Hint**: the constructor that takes a capacity, rather than the one that takes text.


In [ ]:
/// Return an empty `String` that can hold at least `bytes` bytes without reallocating.
///
/// TODO: replace `todo!()` with the reserving constructor.
fn with_room(bytes: usize) -> String {
    todo!()
}

// --- check (do not modify) ---
assert!(with_room(64).capacity() >= 64);
assert_eq!(with_room(16).len(), 0);
assert!(with_room(64).is_empty());
println!("Exercise 5.2 passed");


### Exercise 5.3 — Reserve on an existing buffer

**Goal**: make room for more bytes without disturbing what is already there.

**Info**: `reserve(n)` guarantees room for `n` **more** bytes, so it is measured against `len()`, not against zero. The contents are untouched.

**Objectives**

- ensure `buffer` can take `needed` more bytes without reallocating
- leave its contents unchanged
- keep the signature unchanged

**See**: §5.3 of the example notebook.

**Hint**: the method is not the constructor; capacity arithmetic is relative to the current length.


In [ ]:
/// Ensure `buffer` has room for `needed` more bytes, leaving its contents untouched.
///
/// TODO: replace `todo!()` with the reservation.
fn ensure_capacity(buffer: &mut String, needed: usize) {
    todo!()
}

// --- check (do not modify) ---
let mut buffer = String::from("abc");
ensure_capacity(&mut buffer, 64);
assert!(buffer.capacity() >= buffer.len() + 64);
assert_eq!(buffer, "abc");
assert_eq!(buffer.len(), 3);
println!("Exercise 5.3 passed");


### Exercise 5.4 — Edit in the middle

**Goal**: insert and remove at a byte index.

**Info**: `insert_str`, `insert` and `remove` are byte-indexed and O(n), because everything after the index shifts. `remove` returns the `char` it took out.

**Objectives**

- insert `XY` at byte index 0 of `text`
- then remove the character at the end of `text`
- return the removed character
- keep the signature unchanged

**See**: §5.4 of the example notebook.

**Hint**: the insert takes a slice, the removal returns a value. The index of the last character is not `len()`.


In [ ]:
/// Insert `"XY"` at byte 0 of `text`, remove the final character, and return it.
///
/// TODO: replace `todo!()` with the insert, the removal and the returned value.
fn edit(text: &mut String) -> char {
    todo!()
}

// --- check (do not modify) ---
let mut text = String::from("abc");
assert_eq!(edit(&mut text), 'c');
assert_eq!(text, "XYab");
assert_eq!(text.len(), 4);
println!("Exercise 5.4 passed");


### Exercise 5.5 — In-place editing through a mutable slice

**Goal**: modify a region of a string without reallocating.

**Info**: `get_mut(range)` returns `Option<&mut str>`, which is the safe way into a region. `make_ascii_uppercase` edits bytes in place — and, being ASCII-only, it cannot change the length, which is exactly why it can work in place.

**Objectives**

- make the first byte-range of `text` ASCII-uppercase, using `get_mut` and the in-place method
- leave non-ASCII text untouched: the range `0..1` is not a char boundary inside a multi-byte character, so `get_mut` returns `None`
- keep the signature unchanged

**See**: §5.5 of the example notebook.

**Hint**: `if let Some(region) = ...` — do nothing in the `None` case.


In [ ]:
/// Make the first ASCII byte of `text` uppercase, leaving non-ASCII text unchanged.
///
/// TODO: replace `todo!()` with the in-place edit.
fn uppercase_first_byte(text: &mut String) {
    todo!()
}

// --- check (do not modify) ---
let mut ascii = String::from("abc");
uppercase_first_byte(&mut ascii);
assert_eq!(ascii, "Abc");

let mut accented = String::from("éa");
uppercase_first_byte(&mut accented);
assert_eq!(accented, "éa");
assert_eq!(accented.len(), 3);
println!("Exercise 5.5 passed");


## 6. Iterating: `bytes`, `chars`, `char_indices`

### Exercise 6.1 — Iterate bytes

**Goal**: read a string as the byte buffer it is.

**Info**: `.bytes()` yields `u8` with no decoding at all. It is the right iterator for ASCII protocols, checksums and any wire format defined in bytes.

**Objectives**

- return every byte of `text`, in order
- keep the signature unchanged

**See**: §6.1 of the example notebook.

**Hint**: one iterator adaptor, collected.


In [ ]:
/// Return the bytes of `text`.
///
/// TODO: replace `todo!()` with the collected bytes.
fn byte_values(text: &str) -> Vec<u8> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(byte_values("hi"), vec![104, 105]);
assert_eq!(byte_values("é"), vec![195, 169]);
assert_eq!(byte_values("").len(), 0);
println!("Exercise 6.1 passed");


### Exercise 6.2 — A character pipeline

**Goal**: filter and reorder with the character iterator.

**Info**: `chars()` composes with the iterator toolkit. Reversing is `.chars().rev()`, and collecting `char`s back into a `String` is the inverse of `chars()`. Note that `.bytes().rev()` would produce invalid UTF-8 and therefore cannot be collected into a `String`.

**Objectives**

- keep only the ASCII digits of `text`
- return them in reverse order, as a `String`
- keep the signature unchanged

**See**: §6.2 of the example notebook.

**Hint**: filter, then reverse, then collect — the order of the two adaptors does not matter here.


In [ ]:
/// Return every ASCII digit of `text`, in reverse order.
///
/// TODO: replace `todo!()` with the pipeline.
fn digits_reversed(text: &str) -> String {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(digits_reversed("a1b2c3"), "321");
assert_eq!(digits_reversed("abc"), "");
assert_eq!(digits_reversed("héllo7"), "7");
assert_eq!(digits_reversed("³12"), "21");
println!("Exercise 6.2 passed");


### Exercise 6.3 — Byte offsets versus character offsets

**Goal**: find a subsequence, and convert its position from one unit to the other.

**Info**: `find` reports a **byte** index, which is what slicing needs. A character index has to be counted up to that byte. Coming from Python, where `str.find` reports character indices, this is the first thing that goes wrong.

**Objectives**

- implement `byte_offset` — the byte index of the first occurrence of `needle`
- implement `char_index` — the character index of that same occurrence, counted up to it
- return `None` from both when `needle` does not occur
- keep both signatures unchanged

**See**: §6.3 of the example notebook.

**Hint**: `find` gives you the byte index; the character index is a count over the slice *before* it.


In [ ]:
/// Return the byte index of the first occurrence of `needle`.
///
/// TODO: replace `todo!()` with the search.
fn byte_offset(haystack: &str, needle: &str) -> Option<usize> {
    todo!()
}

/// Return the character index of the first occurrence of `needle`.
///
/// TODO: replace `todo!()` with the byte search plus a character count.
fn char_index(haystack: &str, needle: &str) -> Option<usize> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(byte_offset("héllo", "llo"), Some(3));
assert_eq!(char_index("héllo", "llo"), Some(2));
assert_eq!(byte_offset("banana", "an"), Some(1));
assert_eq!(byte_offset("héllo", "zzz"), None);
assert_eq!(char_index("héllo", "zzz"), None);
println!("Exercise 6.3 passed");


## 7. The Method Inventory

### Exercise 7.1 — The three pattern kinds

**Goal**: use the pattern system that `contains` and friends accept.

**Info**: `contains`, `starts_with` and `find` take a `char`, a `&str`, or a `&[char]` meaning "any of these". That is a trait, not overloading.

**Objectives**

- return `true` when `text` contains any character from `needles`
- return `false` for an empty `needles`
- keep the signature unchanged

**See**: §7.1 of the example notebook.

**Hint**: the slice itself is a valid pattern; no loop is needed.


In [ ]:
/// Return true when `text` contains any of the characters in `needles`.
///
/// TODO: replace `todo!()` with the pattern search.
fn has_any(text: &str, needles: &[char]) -> bool {
    todo!()
}

// --- check (do not modify) ---
assert!(has_any("hello", &['x', 'l']));
assert!(!has_any("hello", &['x', 'y']));
assert!(!has_any("hello", &[]));
assert!(has_any("héllo", &['é']));
println!("Exercise 7.1 passed");


### Exercise 7.2 — Trim a character set

**Goal**: strip a character from both ends.

**Info**: `trim` removes whitespace. The `_matches` family removes a **set of characters**, so `trim_matches('0')` strips every leading and trailing `0` — not the substring `"00"`.

**Objectives**

- strip every leading and trailing `0` from `text`
- keep the signature unchanged

**See**: §7.2 of the example notebook.

**Hint**: one method, applied at both ends at once.


In [ ]:
/// Strip leading and trailing `0` characters from `text`.
///
/// TODO: replace `todo!()` with the trimming.
fn strip_zeros(text: &str) -> &str {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(strip_zeros("0012"), "12");
assert_eq!(strip_zeros("1200"), "12");
assert_eq!(strip_zeros("0000"), "");
assert_eq!(strip_zeros("102"), "102");
println!("Exercise 7.2 passed");


### Exercise 7.3 — Two case mappings, two answers

**Goal**: upper-case text with, and without, the Unicode tables.

**Info**: The Unicode mapping can change the length, because one character can become several. The ASCII mapping cannot, and leaves `ß` alone. For identifiers and protocol tokens — ASCII by specification — the ASCII form is both correct and cheaper.

**Objectives**

- implement `shout_unicode` with the full Unicode mapping
- implement `shout_ascii` with the ASCII-only mapping
- keep both signatures unchanged

**See**: §7.3 of the example notebook.

**Hint**: the two methods differ by one word, `ascii`.


In [ ]:
/// Uppercase `text` with the full Unicode mapping.
///
/// TODO: replace `todo!()` with the Unicode mapping.
fn shout_unicode(text: &str) -> String {
    todo!()
}

/// Uppercase `text` with the ASCII-only mapping.
///
/// TODO: replace `todo!()` with the ASCII mapping.
fn shout_ascii(text: &str) -> String {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(shout_unicode("ß"), "SS");
assert_eq!(shout_ascii("ß"), "ß");
assert_eq!(shout_unicode("grüße"), "GRÜSSE");
assert_eq!(shout_ascii("grüße"), "GRüßE");
assert_eq!(shout_unicode("abc"), "ABC");
println!("Exercise 7.3 passed");


### Exercise 7.4 — Strip and split, without allocating

**Goal**: use the two parsing primitives that return `Option`.

**Info**: `strip_prefix`/`strip_suffix` remove a known affix and return `None` when it is absent. `split_once` splits at the first occurrence of a pattern. All of them borrow from the input, so none of them copies text.

**Objectives**

- implement `file_stem` — `name` without its `.rs` suffix, or `None`
- implement `key_value` — `line` split at its first `=`, or `None`
- keep both signatures unchanged

**See**: §7.4 of the example notebook.

**Hint**: one suffix strip; one first-occurrence split.


In [ ]:
/// Return `name` without a trailing `.rs`, or `None` when it has none.
///
/// TODO: replace `todo!()` with the suffix strip.
fn file_stem(name: &str) -> Option<&str> {
    todo!()
}

/// Split `line` at its first `=`, or return `None` when there is no `=`.
///
/// TODO: replace `todo!()` with the first-occurrence split.
fn key_value(line: &str) -> Option<(&str, &str)> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(file_stem("main.rs"), Some("main"));
assert_eq!(file_stem("main.txt"), None);
assert_eq!(file_stem(".rs"), Some(""));
assert_eq!(key_value("k=v=1"), Some(("k", "v=1")));
assert_eq!(key_value("kv"), None);
assert_eq!(key_value("=v"), Some(("", "v")));
println!("Exercise 7.4 passed");


### Exercise 7.5 — Replace, but only the first few

**Goal**: substitute characters while controlling the count.

**Info**: Both replacement methods allocate, because the result length is unknown up front. `from` and `to` may each be a `char` or a `&str`, independently, so a one-character pattern can be replaced by multi-character text.

**Objectives**

- replace only the **first two** spaces of `text` with underscores
- leave any later space alone
- keep the signature unchanged

**See**: §7.5 of the example notebook.

**Hint**: the bounded variant takes the limit as its last argument.


In [ ]:
/// Replace the first two spaces of `text` with `_`, leaving later spaces alone.
///
/// TODO: replace `todo!()` with the bounded replacement.
fn underscore_first_two(text: &str) -> String {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(underscore_first_two("a b c d"), "a_b_c d");
assert_eq!(underscore_first_two("ab"), "ab");
assert_eq!(underscore_first_two("  "), "__");
assert_eq!(underscore_first_two(""), "");
println!("Exercise 7.5 passed");


### Exercise 7.6 — One buffer for many pieces

**Goal**: build a result without allocating per piece.

**Info**: A `String` can be cleared and refilled, keeping its capacity, and appending a `&str` never copies the source. Reusing one buffer turns a per-piece allocation into one allocation for the whole job.

**Objectives**

- concatenate `pieces` into a single `String` using one reusable buffer
- return the concatenation, with no separator between pieces
- keep the signature unchanged

**See**: §7.6 of the example notebook.

**Hint**: create the buffer once outside the loop; append inside it.


In [ ]:
/// Concatenate `pieces` through a single reusable buffer.
///
/// TODO: replace `todo!()` with the loop over one buffer.
fn join_with_one_buffer(pieces: &[&str]) -> String {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(join_with_one_buffer(&["a", "bb", "ccc"]), "abbccc");
assert_eq!(join_with_one_buffer(&[]), "");
assert_eq!(join_with_one_buffer(&["", "x", ""]), "x");
println!("Exercise 7.6 passed");


### Exercise 7.7 — The ASCII predicates

**Goal**: use the narrow predicate rather than the Unicode one.

**Info**: `is_numeric()` is true for Unicode numeric characters such as `'³'`, which `is_ascii_digit()` rejects. When validating a wire format, the ASCII predicate is the one that matches the specification.

**Objectives**

- return `true` when every character of `text` is an ASCII digit
- return `true` for the empty string, which is vacuously all digits
- keep the signature unchanged

**See**: §7.7 of the example notebook.

**Hint**: an `all` over the characters, with the ASCII-suffixed predicate.


In [ ]:
/// Return true when every character of `text` is an ASCII digit.
///
/// TODO: replace `todo!()` with the scan.
fn all_ascii_digits(text: &str) -> bool {
    todo!()
}

// --- check (do not modify) ---
assert!(all_ascii_digits("20250101"));
assert!(all_ascii_digits(""));
assert!(!all_ascii_digits("12a"));
assert!(!all_ascii_digits("³"));
println!("Exercise 7.7 passed");


### Exercise 7.8 — Characters are numbers, with a checked direction

**Goal**: convert between `char` and its scalar value, in both directions.

**Info**: `char` is a Unicode scalar value, so it converts to `u32` with `as` — always valid. The reverse is **not** total: the UTF-16 surrogate range and anything above `0x10FFFF` are not scalar values, so the checked constructor returns `Option`.

**Objectives**

- implement `char_code` — the scalar value of `c` as a `u32`
- implement `char_from_code` — the character for `code`, or `None` for a surrogate or an out-of-range value
- keep both signatures unchanged

**See**: §7.8 of the example notebook.

**Hint**: `as` for the cheap direction; an associated function starting with `from_` for the checked one.


In [ ]:
/// Return the Unicode scalar value of `c` as a `u32`.
///
/// TODO: replace `todo!()` with the cast.
fn char_code(c: char) -> u32 {
    todo!()
}

/// Return the character for `code`, or `None` when it is not a scalar value.
///
/// TODO: replace `todo!()` with the checked constructor.
fn char_from_code(code: u32) -> Option<char> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(char_code('A'), 65);
assert_eq!(char_code('é'), 233);
assert_eq!(char_code('👋'), 128075);
assert_eq!(char_from_code(0x41), Some('A'));
assert_eq!(char_from_code(0x1F44B), Some('👋'));
assert_eq!(char_from_code(0xD800), None);
assert_eq!(char_from_code(0x110000), None);
println!("Exercise 7.8 passed");


## 8. Splitting, Lines and Joining

### Exercise 8.1 — Split on a fixed separator

**Goal**: parse a delimited line.

**Info**: `split` yields **borrowed** slices of the original, with no allocation, and it keeps empty fields. That is the behaviour a format parser wants, and it is also what surprises people coming from `split_whitespace`.

**Objectives**

- return the comma-separated fields of `line`, empty fields included
- keep the signature unchanged

**See**: §8.1 of the example notebook.

**Hint**: collecting the split iterator into a slice-of-str vector.


In [ ]:
/// Return the comma-separated fields of `line`, keeping empty fields.
///
/// TODO: replace `todo!()` with the split.
fn csv_fields(line: &str) -> Vec<&str> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(csv_fields("a,b"), vec!["a", "b"]);
assert_eq!(csv_fields("a,b,,c").len(), 4);
assert_eq!(csv_fields("a,b,,c")[2], "");
assert_eq!(csv_fields(""), vec![""]);
println!("Exercise 8.1 passed");


### Exercise 8.2 — Whitespace and line endings

**Goal**: tokenise, and count lines in both file formats.

**Info**: `split_whitespace` collapses runs of whitespace and skips empties — a tokenizer, not a parser. `lines()` splits on `\n` and strips a trailing `\r`, which is what makes it correct for Linux and Windows files alike. Neither allocates.

**Objectives**

- implement `word_count` — the number of whitespace-separated words
- implement `line_count` — the number of lines, treating `\r\n` and `\n` alike
- keep both signatures unchanged

**See**: §8.1 and §8.2 of the example notebook.

**Hint**: one counting iterator each. Do not split on `'\n'` by hand.


In [ ]:
/// Count the whitespace-separated words in `text`.
///
/// TODO: replace `todo!()` with the tokenizer count.
fn word_count(text: &str) -> usize {
    todo!()
}

/// Count the lines of `text`, treating `\r\n` and `\n` alike.
///
/// TODO: replace `todo!()` with the line count.
fn line_count(text: &str) -> usize {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(word_count("  a  b\tc "), 3);
assert_eq!(word_count(""), 0);
assert_eq!(word_count("   "), 0);
assert_eq!(line_count("a\r\nb\nc"), 3);
assert_eq!(line_count(""), 0);
assert_eq!(line_count("a"), 1);
println!("Exercise 8.2 passed");


### Exercise 8.3 — Join and repeat

**Goal**: assemble text from pieces in one allocation.

**Info**: `join` is a method on slices of strings, not on `String`. It handles the separators, so it cannot emit one before the first element or after the last — the classic bug in a hand-written loop.

**Objectives**

- join `fields` with `;`
- return `""` for an empty slice
- keep the signature unchanged

**See**: §8.3 of the example notebook.

**Hint**: the separator is the argument; the slice is the receiver.


In [ ]:
/// Join `fields` with `;`.
///
/// TODO: replace `todo!()` with the join.
fn join_fields(fields: &[&str]) -> String {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(join_fields(&["a", "b"]), "a;b");
assert_eq!(join_fields(&["solo"]), "solo");
assert_eq!(join_fields(&["", ""]), ";");
assert_eq!(join_fields(&[]), "");
println!("Exercise 8.3 passed");


## 9. Converting Between Bytes and Text

### Exercise 9.1 — Bytes in, checked

**Goal**: validate bytes as UTF-8, borrowing rather than copying.

**Info**: Every conversion from bytes to text can fail, because not every byte sequence is UTF-8. The borrowed form returns `Result<&str, Utf8Error>` and copies nothing.

**Objectives**

- return the text of `bytes`, or the error
- keep the signature unchanged

**See**: §9.1 of the example notebook.

**Hint**: a function in `std::str`, not a `String` method — it does not allocate, so it has to be an associated function.


In [ ]:
/// Return `bytes` as text, or the UTF-8 error.
///
/// TODO: replace `todo!()` with the checked conversion.
fn text_from_bytes(bytes: &[u8]) -> Result<&str, std::str::Utf8Error> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(text_from_bytes(b"hi"), Ok("hi"));
assert_eq!(text_from_bytes(b""), Ok(""));
assert!(text_from_bytes(&[0xff]).is_err());
assert_eq!(text_from_bytes("é".as_bytes()), Ok("é"));
println!("Exercise 9.1 passed");


### Exercise 9.2 — Read the error

**Goal**: distinguish a corrupt byte from a truncated read.

**Info**: `Utf8Error::valid_up_to()` is the byte index where validation stopped. `error_len()` is `Some(n)` for an invalid sequence, and `None` when the input simply ended in the middle of a multi-byte character — a truncated read, not corruption.

**Objectives**

- return `(valid_up_to, error_len)` of the first UTF-8 error in `bytes`
- keep the signature unchanged

**See**: §9.2 of the example notebook.

**Hint**: produce the error with the checked conversion from §9.1, then call its two accessors.


In [ ]:
/// Return `(valid_up_to, error_len)` of the first UTF-8 error in `bytes`.
///
/// TODO: replace `todo!()` with the error inspection.
fn utf8_report(bytes: &[u8]) -> (usize, Option<usize>) {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(utf8_report(&[0x68, 0xc3]), (1, None));
assert_eq!(utf8_report(&[0x68, 0xff]), (1, Some(1)));
assert_eq!(utf8_report(&[0xff, 0x68]), (0, Some(1)));
println!("Exercise 9.2 passed");


### Exercise 9.3 — The zero-copy round trip

**Goal**: leave text as bytes and come back with a validation.

**Info**: `as_bytes` reinterprets the same buffer for free, and the owned form hands the allocation back without copying. The reverse direction is the one that validates — which is the design: producing UTF-8 from known-good text cannot fail, accepting bytes from outside must be checked.

**Objectives**

- send `text` out through its bytes and back through the owned conversion
- return the text unchanged
- keep the signature unchanged

**See**: §9.3 of the example notebook.

**Hint**: one borrow, one copy to a `Vec`, then the checked owned conversion.


In [ ]:
/// Return `text` after a trip through its bytes and back.
///
/// TODO: replace `todo!()` with the round trip.
fn round_trip(text: &str) -> String {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(round_trip("héllo"), "héllo");
assert_eq!(round_trip(""), "");
assert_eq!(round_trip("👋").len(), 4);
println!("Exercise 9.3 passed");


## 10. Passing Strings: `&str` in, `String` out

### Exercise 10.1 — Accept everything

**Goal**: write a parameter that every string-like value satisfies.

**Info**: A `&str` parameter accepts a literal, a `&String` (by deref coercion) and a slice, with no conversion at the call site. `&String` is strictly less useful and buys nothing.

**Objectives**

- return the byte length of `text`
- keep the signature unchanged — the point is the signature

**See**: §10.1 of the example notebook.

**Hint**: the body is one method call. The exercise is the type of the parameter.


In [ ]:
/// Return the byte length of `text`.
///
/// TODO: replace `todo!()` with the length.
fn byte_len(text: &str) -> usize {
    todo!()
}

// --- check (do not modify) ---
let owned = String::from("deref");
assert_eq!(byte_len("literal"), 7);
assert_eq!(byte_len(&owned), 5);
assert_eq!(byte_len(owned.as_str()), 5);
assert_eq!(byte_len(&owned[0..2]), 2);
assert_eq!(byte_len(""), 0);
println!("Exercise 10.1 passed");


### Exercise 10.2 — Build with `String`, borrow with `&str`

**Goal**: choose the return type from what the function does to the data.

**Info**: A function that creates text must own its result, because a `&str` pointing into a local is a dangling reference — `E0106` — since there is nothing for it to borrow from. A function that merely *reads* the input can return a slice of it, and the elision rule ties that slice's lifetime to the parameter.

**Objectives**

- implement `exclaim` — `text` with `!` appended, as a new owned value
- implement `head_word` — the part of `text` before the first space, borrowed from `text`, or all of `text` when there is no space
- keep both signatures unchanged

**See**: §10.2 of the example notebook.

**Hint**: `exclaim` needs an owned buffer to push into; `head_word` needs a `find` and a slice, or the whole input.


In [ ]:
/// Return `text` with `!` appended, as a new owned value.
///
/// TODO: replace `todo!()` with the owned build.
fn exclaim(text: &str) -> String {
    todo!()
}

/// Return the part of `text` before the first space, borrowed from `text`.
///
/// TODO: replace `todo!()` with the borrow.
fn head_word(text: &str) -> &str {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(exclaim("hello"), "hello!");
assert_eq!(exclaim(""), "!");
assert_eq!(head_word("hello world"), "hello");
assert_eq!(head_word("hello"), "hello");
assert_eq!(head_word(""), "");
println!("Exercise 10.2 passed");


## 11. Text Processing Patterns

### Exercise 11.1 — Split, trim, parse, with `?`

**Goal**: express "well-formed or no result" in one line.

**Info**: Each `?` in an `Option`-returning function is a short-circuit: a missing separator, or a value that does not parse, yields `None`. That is why `split_once` returns an `Option` rather than panicking.

**Objectives**

- split `line` at its first `=`
- trim both sides, and parse the value as an `i64`
- return `None` when either step fails
- keep the signature unchanged

**See**: §11.1 of the example notebook.

**Hint**: `?` twice. `parse` then `ok` converts the `Result` into the `Option` that `?` can propagate here.


In [ ]:
/// Parse `<key>=<value>` with both sides trimmed, or return `None`.
///
/// TODO: replace `todo!()` with the `?` chain.
fn parse_pair(line: &str) -> Option<(&str, i64)> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(parse_pair("  latency_ns = 250  "), Some(("latency_ns", 250)));
assert_eq!(parse_pair("k = -1"), Some(("k", -1)));
assert_eq!(parse_pair("broken"), None);
assert_eq!(parse_pair("k=x"), None);
println!("Exercise 11.1 passed");


### Exercise 11.2 — A lazy pipeline over words

**Goal**: walk text once and allocate nothing intermediate.

**Info**: `split_whitespace` yields borrowed slices, and the iterator adaptors consume them without building collections. Compare with the Python idiom, which materialises a list of words and then a list of lengths.

**Objectives**

- implement `longest_word` — the longest whitespace-separated word, borrowed from `text`, or `""` for empty input
- implement `total_bytes` — the summed byte length of all the words
- keep both signatures unchanged

**See**: §11.2 of the example notebook.

**Hint**: `max_by_key` over the words for the first; `map` then `sum` for the second. Both stay lazy until consumed.


In [ ]:
/// Return the longest whitespace-separated word of `text`, or `""`.
///
/// TODO: replace `todo!()` with the borrowed maximum.
fn longest_word(text: &str) -> &str {
    todo!()
}

/// Return the summed byte length of every whitespace-separated word of `text`.
///
/// TODO: replace `todo!()` with the one-pass sum.
fn total_bytes(text: &str) -> usize {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(longest_word("the quickest brown"), "quickest");
assert_eq!(longest_word(""), "");
assert_eq!(longest_word("one"), "one");
assert_eq!(total_bytes("ab cde"), 5);
assert_eq!(total_bytes("   "), 0);
println!("Exercise 11.2 passed");


### Exercise 11.3 — Normalise a key, or just compare one

**Goal**: pick the allocating and non-allocating forms of the same idea.

**Info**: Configuration keys and header names are ASCII by specification and are compared case-insensitively. Storing a normalised key costs an allocation; comparing in place costs nothing. Choose per use, not per habit.

**Objectives**

- implement `normalise_key` — `raw` trimmed and ASCII-lowercased
- implement `same_key` — `true` when `a` and `b` are equal ignoring ASCII case, with no allocation
- keep both signatures unchanged

**See**: §11.3 of the example notebook.

**Hint**: the allocating one is a chain of two methods; the in-place one is a single comparison method whose name contains `ascii_case`.


In [ ]:
/// Return `raw` trimmed and ASCII-lowercased.
///
/// TODO: replace `todo!()` with the normalising chain.
fn normalise_key(raw: &str) -> String {
    todo!()
}

/// Return true when `a` and `b` match ignoring ASCII case.
///
/// TODO: replace `todo!()` with the in-place comparison.
fn same_key(a: &str, b: &str) -> bool {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(normalise_key("  Content-Type  "), "content-type");
assert_eq!(normalise_key(""), "");
assert!(same_key("Content-Type", "content-type"));
assert!(!same_key("a", "b"));
assert!(same_key("", ""));
println!("Exercise 11.3 passed");


### Exercise 11.4 — Count in one pass

**Goal**: histogram a delimited field.

**Info**: `entry(...).or_insert(0)` updates a count in one lookup, and the keys can borrow from the source string, so no string is copied for a key. A `HashMap` iterates in an order you cannot rely on, so anything that compares the result must sort it first.

**Objectives**

- count each non-empty comma-separated event in `events`
- return the `(event, count)` pairs **sorted by event name**
- keep the signature unchanged

**See**: §11.4 of the example notebook.

**Hint**: `split`, an entry-based count, then collect and sort.


In [ ]:
/// Count each non-empty comma-separated event, sorted by event name.
///
/// TODO: replace `todo!()` with the histogram and the sort.
fn event_counts(events: &str) -> Vec<(&str, u32)> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(event_counts("trade,quote,trade"), vec![("quote", 1), ("trade", 2)]);
assert_eq!(event_counts("a,,a"), vec![("a", 2)]);
assert_eq!(event_counts(""), vec![]);
println!("Exercise 11.4 passed");


## 12. Formatting and Display

### Exercise 12.1 — Append formatted text

**Goal**: build a formatted `String` through the `fmt::Write` trait.

**Info**: `write!` shares its format language with `println!` but writes into any `fmt::Write`, including a `String`. For a `String` it is infallible, though it still returns a `Result`, because the trait also covers I/O-backed writers.

**Objectives**

- return `"<label> = <value>"`, for example `latency = 250`
- build it with `write!` into a `String` rather than with `format!`
- keep the signature unchanged

**See**: §12.1 of the example notebook.

**Hint**: the trait has to be in scope for the method to exist — `use std::fmt::Write as _;` is already in the cell. The write returns a `Result`; this writer cannot fail.


In [ ]:
use std::fmt::Write as _;

/// Return `"<label> = <value>"`.
///
/// TODO: replace `todo!()` with a `String` built by `write!`.
fn format_row(label: &str, value: i32) -> String {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(format_row("latency", 250), "latency = 250");
assert_eq!(format_row("", 0), " = 0");
assert_eq!(format_row("a", -1), "a = -1");
println!("Exercise 12.1 passed");


### Exercise 12.2 — Alignment, padding and radix

**Goal**: use the format specification rather than hand-rolled padding.

**Info**: The spec is `{[argument][:[fill][align][sign][#][0][width][.precision][type]]}`. A width may be given by a named argument (`{value:>width$}`), which is how a column width that is only known at runtime is expressed. `{:#x}` adds the prefix; `{:08x}` zero-pads.

**Objectives**

- implement `right_aligned` — `value` right-aligned in a field of `width` columns
- implement `hex_padded` — `value` as `0x` plus 8 zero-padded lowercase hex digits
- keep both signatures unchanged

**See**: §12.2 of the example notebook.

**Hint**: alignment is one character before the width; the width itself may be an argument with a `$`.


In [ ]:
/// Return `value` right-aligned in a field of `width` columns.
///
/// TODO: replace `todo!()` with the formatted value.
fn right_aligned(value: i32, width: usize) -> String {
    todo!()
}

/// Return `value` as `0x` plus 8 zero-padded lowercase hex digits.
///
/// TODO: replace `todo!()` with the formatted value.
fn hex_padded(value: u32) -> String {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(right_aligned(42, 5), "   42");
assert_eq!(right_aligned(42, 2), "42");
assert_eq!(right_aligned(-7, 4), "  -7");
assert_eq!(hex_padded(255), "0x000000ff");
assert_eq!(hex_padded(0), "0x00000000");
println!("Exercise 12.2 passed");


### Exercise 12.3 — Display versus Debug

**Goal**: render a value for a human, or render it unambiguously.

**Info**: For a `&str` the two traits differ visibly: `Display` prints the text, `Debug` prints it quoted with its escapes, so a tab is visible as `\t`. `Debug` is what an assertion message or a log line wants.

**Objectives**

- return the `Debug` form of `text` when `debug` is true
- return the `Display` form otherwise
- keep the signature unchanged

**See**: §12.3 of the example notebook.

**Hint**: the two arms of the `if` are two format strings that differ by one character after the colon.


In [ ]:
/// Return `text` in `Debug` form when `debug` is true, `Display` form otherwise.
///
/// TODO: replace `todo!()` with the formatted text.
fn render(text: &str, debug: bool) -> String {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(render("a\tb", true), "\"a\\tb\"");
assert_eq!(render("a\tb", false), "a\tb");
assert_eq!(render("", true), "\"\"");
assert_eq!(render("é", false), "é");
println!("Exercise 12.3 passed");


### Exercise 12.4 — Join, then append a formatted suffix

**Goal**: combine the two idioms that beat manual concatenation.

**Info**: `join` handles separators in one allocation, and `write!` appends a formatted piece. Neither can emit a separator in the wrong place, which is the bug a hand-written loop usually has.

**Objectives**

- return `fields` joined with `;`, then ` | orders=<count>` appended
- for an empty `fields`, the line still ends with the suffix, so it starts with a space
- keep the signature unchanged

**See**: §12.4 of the example notebook.

**Hint**: one `join` for the fields, then a formatted suffix — either `format!` around both parts or a `write!` into the joined string.


In [ ]:
/// Return `fields` joined by `;` with ` | orders=<count>` appended.
///
/// TODO: replace `todo!()` with the join plus the formatted suffix.
fn order_line(fields: &[&str], count: usize) -> String {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(order_line(&["BTC-USD", "buy", "0.5"], 1), "BTC-USD;buy;0.5 | orders=1");
assert_eq!(order_line(&["x"], 0), "x | orders=0");
assert_eq!(order_line(&[], 7), " | orders=7");
println!("Exercise 12.4 passed");


## 13. Reading the Compiler on Strings

### Exercise 13.1 — The legal spellings

**Goal**: reach for a byte or a character without the illegal `s[0]`.

**Info**: Two diagnostics dominate this chapter: `E0277` when you index a `str` with an integer, and `E0308` when types that look compatible are not. The compiler's note for `E0277` names the two routes directly — the byte route, and the one that decodes.

**Objectives**

- implement `byte_at` — the byte at `index`, treating `text` as bytes
- implement `char_at` — the character at **character** index `index`, or `None`
- keep both signatures unchanged

**See**: §13.1 of the example notebook.

**Hint**: one is an index expression on the byte view; the other is the walk from §2.4.


In [ ]:
/// Return the byte at `index`, treating `text` as bytes.
///
/// TODO: replace `todo!()` with the byte access.
fn byte_at(text: &str, index: usize) -> u8 {
    todo!()
}

/// Return the character at character index `index`, or `None`.
///
/// TODO: replace `todo!()` with the character lookup.
fn char_at(text: &str, index: usize) -> Option<char> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(byte_at("hi", 0), 104);
assert_eq!(byte_at("é", 0), 195);
assert_eq!(char_at("héllo", 0), Some('h'));
assert_eq!(char_at("héllo", 1), Some('é'));
assert_eq!(char_at("hi", 5), None);
println!("Exercise 13.1 passed");


## 14. Unicode Beyond ASCII

### Exercise 14.1 — Bytes, characters, and neither is "what the user sees"

**Goal**: report both widths of a string, and notice how far apart they get.

**Info**: A `char` is a scalar value, not a user-perceived character. A precomposed `é` is one scalar value; the same visible character written as `e` plus a combining accent is two; an emoji with a skin-tone modifier is two. Grapheme clusters need Unicode tables that `std` does not ship, which is why `unicode-segmentation` exists — and why this series stays with bytes and scalar values.

**Objectives**

- return `(byte length, character count)` of `text`
- keep the signature unchanged

**See**: §14.1 of the example notebook.

**Hint**: `len` is free, the other is a walk — one tuple, both calls.


In [ ]:
/// Return `(byte length, character count)` of `text`.
///
/// TODO: replace `todo!()` with the pair.
fn widths(text: &str) -> (usize, usize) {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(widths(""), (0, 0));
assert_eq!(widths("abc"), (3, 3));
assert_eq!(widths("é"), (2, 1));
assert_eq!(widths("e\u{301}"), (3, 2));
assert_eq!(widths("👋🏽"), (8, 2));
println!("Exercise 14.1 passed");


### Exercise 14.2 — Case mapping can add characters

**Goal**: measure the length change instead of assuming there is none.

**Info**: Upper-casing is a length-changing operation in the Unicode direction: `ß` becomes `SS`, so the character count grows even though the byte count happens to stay the same. The ASCII variant cannot add or remove anything.

**Objectives**

- implement `uppercase_char_count` — the character count after the Unicode mapping
- implement `ascii_uppercase_char_count` — the character count after the ASCII mapping
- keep both signatures unchanged

**See**: §14.2 of the example notebook.

**Hint**: map, then count, in each case; the mapped value is a `String`.


In [ ]:
/// Return the character count of `text` after the Unicode uppercase mapping.
///
/// TODO: replace `todo!()` with the mapped count.
fn uppercase_char_count(text: &str) -> usize {
    todo!()
}

/// Return the character count of `text` after the ASCII uppercase mapping.
///
/// TODO: replace `todo!()` with the mapped count.
fn ascii_uppercase_char_count(text: &str) -> usize {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(uppercase_char_count(""), 0);
assert_eq!(uppercase_char_count("abc"), 3);
assert_eq!(uppercase_char_count("ß"), 2);
assert_eq!(uppercase_char_count("grüße"), 6);
assert_eq!(ascii_uppercase_char_count("ß"), 1);
assert_eq!(ascii_uppercase_char_count("grüße"), 5);
println!("Exercise 14.2 passed");


### Exercise 14.3 — Validate and encode scalar values

**Goal**: check a code point from a protocol, and encode one for UTF-16 output.

**Info**: `char::from_u32` rejects the surrogate range `0xD800..=0xDFFF` and anything above `0x10FFFF`, returning `None` — it never panics, which makes it the right validation for untrusted input. `encode_utf16` fills a two-slot buffer and returns the slice it used, because a character above the BMP needs a surrogate pair.

**Objectives**

- implement `is_scalar_value` — `true` when `code` is a valid Unicode scalar value
- implement `utf16_units` — the UTF-16 encoding of `c` as a vector of units
- keep both signatures unchanged

**See**: §14.3 of the example notebook.

**Hint**: the checked constructor for the first; a two-element buffer and the encoding method for the second.


In [ ]:
/// Return true when `code` is a valid Unicode scalar value.
///
/// TODO: replace `todo!()` with the validation.
fn is_scalar_value(code: u32) -> bool {
    todo!()
}

/// Return the UTF-16 encoding of `c` as a vector of units.
///
/// TODO: replace `todo!()` with the encoding.
fn utf16_units(c: char) -> Vec<u16> {
    todo!()
}

// --- check (do not modify) ---
assert!(is_scalar_value(0x41));
assert!(is_scalar_value(0x1F44B));
assert!(!is_scalar_value(0xD800));
assert!(!is_scalar_value(0x110000));
assert_eq!(utf16_units('a'), vec![97]);
assert_eq!(utf16_units('é'), vec![233]);
assert_eq!(utf16_units('👋'), vec![55357, 56395]);
println!("Exercise 14.3 passed");


## 15. Performance Notes for Low-Latency Code

### Exercise 15.1 — Reuse one buffer across messages

**Goal**: make a per-message allocation into a per-batch allocation.

**Info**: `clear()` sets the length to zero and keeps the capacity, so a reused buffer stops reallocating after the first message. The parsing itself should stay in borrowed `&str` territory, as in §11.

**Objectives**

- for each message in `messages`, clear and refill a single buffer, then record `(key length, value length)` from the first `=`
- return one pair per message, in input order
- keep the signature unchanged

**See**: §15.1 of the example notebook.

**Hint**: the buffer is created once, outside the loop; `split_once` on the filled buffer gives the two lengths without allocating either key or value.


In [ ]:
/// Parse each message through one reusable buffer, recording `(key length, value length)`.
///
/// TODO: replace `todo!()` with the reuse loop.
fn parse_all(messages: &[&str]) -> Vec<(usize, usize)> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(parse_all(&["a=1", "bb=22", "ccc=333"]), vec![(1, 1), (2, 2), (3, 3)]);
assert_eq!(parse_all(&[]), vec![]);
assert_eq!(parse_all(&["k="]), vec![(1, 0)]);
println!("Exercise 15.1 passed");


### Exercise 15.2 — Stay at the byte level

**Goal**: validate ASCII input without decoding it.

**Info**: `as_bytes()` is free — it reinterprets the same buffer. A byte loop never decodes UTF-8 and never consults a Unicode table, which on a hot path is the difference between a comparison and an allocation.

**Objectives**

- implement `token_bytes` — the bytes of `text`, borrowed, with no copy
- implement `all_ascii_digit_bytes` — `true` when every **byte** is an ASCII digit, including for the empty string
- keep both signatures unchanged

**See**: §15.2 of the example notebook.

**Hint**: the borrowed view plus an `all` over `u8` with the ASCII-suffixed predicate.


In [ ]:
/// Return the bytes of `text` without copying.
///
/// TODO: replace `todo!()` with the byte view.
fn token_bytes(text: &str) -> &[u8] {
    todo!()
}

/// Return true when every byte of `text` is an ASCII digit.
///
/// TODO: replace `todo!()` with the byte scan.
fn all_ascii_digit_bytes(text: &str) -> bool {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(token_bytes("BUY"), b"BUY");
assert_eq!(token_bytes("é"), "é".as_bytes());
assert_eq!(token_bytes("é")[1], 169);
assert!(all_ascii_digit_bytes("20250101"));
assert!(all_ascii_digit_bytes(""));
assert!(!all_ascii_digit_bytes("12a"));
assert!(!all_ascii_digit_bytes("é"));
println!("Exercise 15.2 passed");


### Exercise 15.3 — Borrow when you can, own when you must

**Goal**: write a normaliser that allocates only for input that needs it.

**Info**: `Cow<'_, str>` is "either a borrow or an owned value". Both arms of an `if` have the same type, so the caller writes one code path and an allocation happens only on the inputs that actually needed rewriting.

**Objectives**

- return a borrowed `Cow` when `raw` contains no ASCII uppercase byte
- return an owned, ASCII-lowercased `Cow` otherwise
- keep the signature unchanged

**See**: §15.3 of the example notebook.

**Hint**: the `Borrowed` arm hands back the parameter; the `Owned` arm hands back the lowercased copy.


In [ ]:
/// Lowercase `raw` only when it contains an ASCII uppercase byte; otherwise borrow it.
///
/// TODO: replace `todo!()` with the two `Cow` arms.
fn normalise(raw: &str) -> std::borrow::Cow<'_, str> {
    todo!()
}

// --- check (do not modify) ---
assert!(matches!(normalise("btc-usd"), std::borrow::Cow::Borrowed(_)));
assert!(matches!(normalise("BTC-USD"), std::borrow::Cow::Owned(_)));
assert_eq!(normalise("BTC-USD"), "btc-usd");
assert_eq!(normalise(""), "");
println!("Exercise 15.3 passed");


### Exercise 15.4 — Offsets in bytes, indices in characters

**Goal**: produce both kinds of position for the same character.

**Info**: A byte offset is O(1) once you have it, and it is what slicing needs. A character index is O(n), because it is a walk. Mixing the two units up is the single most common string bug, so it is worth being able to produce each deliberately.

**Objectives**

- implement `byte_offset_of` — the byte offset of the first `target` in `text`
- implement `char_index_of` — the character index of that same occurrence
- return `None` from both when `target` does not occur
- keep both signatures unchanged

**See**: §15.4 of the example notebook.

**Hint**: `char_indices` gives the byte offset for free; `chars().position` gives the character index in one walk.


In [ ]:
/// Return the byte offset of the first `target` in `text`.
///
/// TODO: replace `todo!()` with the offset lookup.
fn byte_offset_of(text: &str, target: char) -> Option<usize> {
    todo!()
}

/// Return the character index of the first `target` in `text`.
///
/// TODO: replace `todo!()` with the index lookup.
fn char_index_of(text: &str, target: char) -> Option<usize> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(byte_offset_of("héllo", 'l'), Some(3));
assert_eq!(char_index_of("héllo", 'l'), Some(2));
assert_eq!(byte_offset_of("héllo", 'h'), Some(0));
assert_eq!(byte_offset_of("héllo", 'z'), None);
assert_eq!(char_index_of("héllo", 'z'), None);
println!("Exercise 15.4 passed");


## 16. Review: Mixed Problems

### Exercise 16.1 — A strict boolean field

**Goal**: parse a field the way a wire protocol defines it.

**Info**: A protocol field is ASCII, padded, and case-insensitive. Normalising it for storage allocates; comparing it in place does not. Here the input is transient, so the in-place form is the right one.

**Objectives**

- trim `field` and compare it ignoring ASCII case
- accept exactly `true` and `false`, and return `None` for anything else
- keep the signature unchanged

**See**: §11.3 and §7.4 of the example notebook.

**Hint**: an `if`/`else if` on two in-place comparisons, with a `None` fallthrough.


In [ ]:
/// Parse `TRUE`/`FALSE` (any ASCII case, optional surrounding spaces) or return `None`.
///
/// TODO: replace `todo!()` with the two-way comparison.
fn parse_bool_field(field: &str) -> Option<bool> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(parse_bool_field(" TRUE "), Some(true));
assert_eq!(parse_bool_field("false"), Some(false));
assert_eq!(parse_bool_field("True"), Some(true));
assert_eq!(parse_bool_field("yes"), None);
assert_eq!(parse_bool_field(""), None);
println!("Exercise 16.1 passed");


### Exercise 16.2 — Summarise a key/value line

**Goal**: chain the parsing primitives end to end.

**Info**: `split_once`, `trim`, `parse` and the ASCII case mapping compose into the shape of a log summariser: everything borrowed until the final `format!`, and every failure mode collapsed into one `None`.

**Objectives**

- parse `<key>=<value>` with both sides trimmed
- upper-case the key with the **ASCII** mapping, parse the value as an `i64`
- return `"<KEY>:<value>"` as an owned `String`, or `None` on any failure
- keep the signature unchanged

**See**: §11.1 and §7.3 of the example notebook.

**Hint**: the first half is §11.1's `?` chain; remember `.parse().ok()?` to turn the parse failure into `None`.


In [ ]:
/// Return `"<KEY>:<value>"` for a `key=value` line, or `None` when it is malformed.
///
/// TODO: replace `todo!()` with the parse-and-format chain.
fn summarise_line(line: &str) -> Option<String> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(summarise_line(" latency_ns = 250 "), Some("LATENCY_NS:250".to_string()));
assert_eq!(summarise_line("k=-1"), Some("K:-1".to_string()));
assert_eq!(summarise_line("bad"), None);
assert_eq!(summarise_line("k=x"), None);
println!("Exercise 16.2 passed");


### Exercise 16.3 — Truncate without corrupting anything

**Goal**: cut text to a limit that is safe on Unicode.

**Info**: Cutting by bytes panics when the cut lands inside a character, and cutting with `&text[..n]` is therefore only safe for ASCII. Cutting by characters cannot panic; it is a walk plus a collect. The formatter's `.precision` does the same thing for `String` arguments, but not for the values you return.

**Objectives**

- return at most `limit` characters of `text`
- never split a character, and never panic
- return all of `text` when it is already short enough
- keep the signature unchanged

**See**: §14.1 and §12.2 of the example notebook.

**Hint**: `take` on the character iterator, then collect back into a `String`.


In [ ]:
/// Return at most `limit` characters of `text`, never splitting one.
///
/// TODO: replace `todo!()` with the character-wise truncation.
fn truncate_chars(text: &str, limit: usize) -> String {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(truncate_chars("héllo", 3), "hél");
assert_eq!(truncate_chars("héllo", 99), "héllo");
assert_eq!(truncate_chars("abc", 0), "");
assert_eq!(truncate_chars("👋🏽", 1), "👋");
println!("Exercise 16.3 passed");


### What Comes Next

Chapter 04's exercises all circled one question: who owns the bytes? A `String` owns them, a `&str` borrows them, and every method that returns `Option<&str>` is handing back a view into someone else's buffer. That is the subject of the next chapter.

| Chapter | Title | What it settles |
|---|---|---|
| **05** | **Ownership and Borrowing** | Moves, copies, borrows, slices and lifetimes — including why the live-slice-then-mutate case in §3.4 is a rule and not a quirk |
| 06 | Native Collections | `Vec`, `HashMap`, `HashSet`, `VecDeque`, `BTreeMap`, and the iterator methods the string pipelines in §6 and §11 already used |
| 07 | I/O and Files | Reading, writing, buffering, and error propagation with `Result` — where `lines()`, `BufReader` and `starts_with` meet |

**Carry forward.** You should now be able to do these without looking anything up:

- choose between `&str` and `String` in a signature, and say why
- report a string's width in bytes, in characters, or as `(byte offset, char)` pairs
- slice by byte offsets, and reach for `get` when the offsets come from outside the program
- convert bytes to text at the right level of strictness, and read a `Utf8Error`
- split, trim, normalise and rejoin text without leaking allocations
- read `E0277`, `E0308` and `E0502` on strings and name the fix each one wants

If any of those still needs looking up, §16 of the example notebook is the one-page answer, and re-running the matching exercise here is the check.
